# Rebuttal Analysis: Process-Level Error Breakdown + Capability Taxonomy
This notebook extracts:
1. **Process-level error types** from OSCAR trajectories and baseline results (for R2-W2)
2. **Capability taxonomy** from question text (for R2-W3)

In [ ]:
import os, json, re, math, glob
import pandas as pd
from collections import Counter, defaultdict

ROOT = '/Users/chiahsiang1/Documents/univearth_arr'
OSCAR_DIR = os.path.join(ROOT, 'results_2602/oscar/python/gemini-2.5-pro+gemini-2.5-flash__v2')
BASELINE_DIR = os.path.join(ROOT, 'results_2511/zero_shot/python/gemini-2.5-pro+gemini-2.5-flash__v1')
DATASET_PATH = os.path.join(ROOT, 'dataset/2026_acl_univearth_1123.csv')

def safe_load_json(fpath):
    with open(fpath) as f:
        text = f.read()
    text = text.replace('NaN', 'null')
    return json.loads(text)

print(f'OSCAR files: {len(os.listdir(OSCAR_DIR))}')
print(f'Baseline files: {len(os.listdir(BASELINE_DIR))}')

## Part 1: Process-Level Error Analysis (OSCAR trajectories)

In [ ]:
# Classify OSCAR errors by *where* in the pipeline they occur
# We look at the trajectory to identify:
#   - plan quality issues (wrong dataset, wrong band)
#   - code syntax / runtime errors (D)
#   - empty collection (C1) -> temporal/spatial mismatch
#   - no valid pixels (C2) -> masking/preprocessing issue
#   - calculation failure (C3) -> numerical issue
#   - wrong answer (A/B but incorrect) -> reasoning/threshold error

def classify_oscar_error(data):
    """Classify the error type from an OSCAR memory JSON."""
    summary = data.get('summary', {})
    trajectory = data.get('trajectory', [])
    metadata = data.get('metadata', {})

    final_answer = summary.get('final_answer', 'D')
    gt_raw = metadata.get('answer')

    # Normalize ground truth
    if gt_raw is None or (isinstance(gt_raw, float) and math.isnan(gt_raw)):
        gt = 'Yes'
    elif str(gt_raw).strip().lower() == 'no':
        gt = 'No'
    else:
        gt = str(gt_raw).strip() if str(gt_raw).strip() not in ('NaN','nan','null','') else 'Yes'

    # Check correctness
    correct = (gt == 'Yes' and final_answer == 'A') or (gt == 'No' and final_answer == 'B')

    if correct:
        return 'Correct', 'correct'

    # Error categorization
    if final_answer == 'D':
        # Look at exec_stderr for more detail
        error_detail = 'syntax/runtime'
        for t in trajectory:
            stderr = t.get('data', {}).get('exec_stderr', '')
            exec_msg = t.get('data', {}).get('exec_msg', '')
            combined = stderr + ' ' + exec_msg
            if 'did not match any bands' in combined or 'band' in combined.lower():
                error_detail = 'wrong_band'
                break
            elif 'not found' in combined.lower() or 'no such' in combined.lower():
                error_detail = 'wrong_dataset'
                break
            elif 'timeout' in combined.lower() or 'timed out' in combined.lower():
                error_detail = 'timeout'
                break
            elif 'quota' in combined.lower() or 'limit' in combined.lower():
                error_detail = 'quota_limit'
                break
        return 'Execution Error (D)', error_detail

    if final_answer == 'C1':
        return 'Empty Collection (C1)', 'temporal_spatial_mismatch'
    if final_answer == 'C2':
        return 'No Valid Pixels (C2)', 'masking_preprocessing'
    if final_answer == 'C3':
        return 'Calculation Failure (C3)', 'numerical_issue'

    if final_answer in ('A', 'B'):
        return 'Wrong Answer', 'reasoning_threshold'

    return 'Unknown', 'unknown'

# Process all OSCAR results
oscar_records = []
for fname in sorted(os.listdir(OSCAR_DIR)):
    if not fname.endswith('.json'):
        continue
    data = safe_load_json(os.path.join(OSCAR_DIR, fname))
    category, detail = classify_oscar_error(data)
    replan_count = data.get('summary', {}).get('replan_attempts', 0)
    oscar_records.append({
        'filename': fname,
        'category': category,
        'detail': detail,
        'replan_attempts': replan_count,
        'final_answer': data.get('summary', {}).get('final_answer', 'D'),
        'question': data.get('metadata', {}).get('question', ''),
    })

df_oscar = pd.DataFrame(oscar_records)
print(f'Total OSCAR results: {len(df_oscar)}')
print()
print('=== Error Category Breakdown ===')
cat_counts = df_oscar['category'].value_counts()
for cat, cnt in cat_counts.items():
    print(f'  {cat:30s}: {cnt:4d} ({cnt/len(df_oscar)*100:.1f}%)')

print()
print('=== Detailed Error Type Breakdown ===')
detail_counts = df_oscar['detail'].value_counts()
for det, cnt in detail_counts.items():
    print(f'  {det:30s}: {cnt:4d} ({cnt/len(df_oscar)*100:.1f}%)')

In [ ]:
# Same analysis for baseline (zero-shot, no reflexion)
baseline_records = []
for fname in sorted(os.listdir(BASELINE_DIR)):
    if not fname.endswith('.json'):
        continue
    data = safe_load_json(os.path.join(BASELINE_DIR, fname))
    metadata = data.get('metadata', {})
    data_list = data.get('data', [])
    if not data_list:
        continue

    zs_answer = data_list[0].get('answer', 'D')
    gt_raw = metadata.get('answer')

    if gt_raw is None or (isinstance(gt_raw, float) and math.isnan(gt_raw)):
        gt = 'Yes'
    elif str(gt_raw).strip().lower() == 'no':
        gt = 'No'
    else:
        gt = str(gt_raw).strip() if str(gt_raw).strip() not in ('NaN','nan','null','') else 'Yes'

    correct = (gt == 'Yes' and zs_answer == 'A') or (gt == 'No' and zs_answer == 'B')

    # Check error details from exec output
    exec_msg = data_list[0].get('exec_msg', '')
    exec_stderr = data_list[0].get('exec_stderr', '') if 'exec_stderr' in data_list[0] else ''
    raw_code = data_list[0].get('raw_code', '') or data_list[0].get('code', '')

    if correct:
        category, detail = 'Correct', 'correct'
    elif zs_answer == 'D':
        detail = 'syntax/runtime'
        combined = str(exec_stderr) + ' ' + str(exec_msg) + ' ' + str(raw_code)
        if 'did not match any bands' in combined or ('band' in combined.lower() and 'error' in combined.lower()):
            detail = 'wrong_band'
        elif 'not found' in combined.lower() or 'Asset' in combined:
            detail = 'wrong_dataset'
        category = 'Execution Error (D)'
    elif zs_answer == 'C1':
        category, detail = 'Empty Collection (C1)', 'temporal_spatial_mismatch'
    elif zs_answer == 'C2':
        category, detail = 'No Valid Pixels (C2)', 'masking_preprocessing'
    elif zs_answer == 'C3':
        category, detail = 'Calculation Failure (C3)', 'numerical_issue'
    elif zs_answer in ('A', 'B'):
        category, detail = 'Wrong Answer', 'reasoning_threshold'
    else:
        category, detail = 'Unknown', 'unknown'

    baseline_records.append({
        'filename': fname,
        'category': category,
        'detail': detail,
        'final_answer': zs_answer,
        'question': metadata.get('question', ''),
    })

df_baseline = pd.DataFrame(baseline_records)
print(f'Total Baseline results: {len(df_baseline)}')
print()
print('=== Baseline Error Category Breakdown ===')
cat_counts_b = df_baseline['category'].value_counts()
for cat, cnt in cat_counts_b.items():
    print(f'  {cat:30s}: {cnt:4d} ({cnt/len(df_baseline)*100:.1f}%)')

In [ ]:
# Side-by-side comparison: Baseline (zero-shot) vs OSCAR
print('=' * 70)
print(f'{"Category":30s} | {"Baseline":>10s} | {"OSCAR":>10s} | {"Delta":>10s}')
print('-' * 70)
all_cats = ['Correct', 'Wrong Answer', 'Empty Collection (C1)', 'No Valid Pixels (C2)', 'Calculation Failure (C3)', 'Execution Error (D)']
for cat in all_cats:
    b_cnt = cat_counts_b.get(cat, 0)
    o_cnt = cat_counts.get(cat, 0)
    b_pct = b_cnt / len(df_baseline) * 100 if len(df_baseline) > 0 else 0
    o_pct = o_cnt / len(df_oscar) * 100 if len(df_oscar) > 0 else 0
    delta = o_pct - b_pct
    print(f'  {cat:28s} | {b_pct:8.1f}% | {o_pct:8.1f}% | {delta:+8.1f}%')
print('=' * 70)

## Part 2: Capability Taxonomy Analysis

In [ ]:
# Load dataset and classify questions by required EO capabilities
df_data = pd.read_csv(DATASET_PATH)
print(f'Total questions: {len(df_data)}')
print(f'Columns: {list(df_data.columns)}')
print(f'Tags: {df_data["Tag"].value_counts().to_dict()}')
print()
print('Sample questions:')
for i, row in df_data.head(5).iterrows():
    print(f'  [{row["Tag"]}] {row["Question"][:120]}')

In [ ]:
# Classify each question by required reasoning capabilities
# These are inferred from question text patterns

def classify_capabilities(question):
    """Infer required EO capabilities from question text."""
    q = question.lower()
    caps = []

    # Temporal reasoning
    temporal_patterns = [
        r'\b(between|from)\s+\w+\s+\d{4}\s+(and|to)',
        r'\b(january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}',
        r'\b\d{4}[-/]\d{2}',
        r'\b(before|after|during|since|prior to)\b',
        r'\b(increase|decrease|change|trend|compared to|higher than|lower than|more than|less than)\b',
    ]
    if any(re.search(p, q) for p in temporal_patterns):
        caps.append('Temporal Reasoning')

    # Spatial reasoning
    spatial_patterns = [
        r'\b(region|area|boundary|country|city|state|province|basin|coast|ocean|lake|river)\b',
        r'\b(north|south|east|west|central|northern|southern|eastern|western)\b',
        r'\b(spatial|geographic|geograph)\b',
    ]
    if any(re.search(p, q) for p in spatial_patterns):
        caps.append('Spatial Reasoning')

    # Spectral index computation
    index_patterns = [
        r'\b(ndvi|ndwi|mndwi|ndsi|evi|savi|nbr|bsi|lst|nir|swir)\b',
        r'\b(band ratio|spectral|reflectance|radiance|emissivity)\b',
        r'\b(surface temperature|land surface|sea surface)\b',
    ]
    if any(re.search(p, q) for p in index_patterns):
        caps.append('Spectral Index Computation')

    # Quantitative comparison
    comparison_patterns = [
        r'\b(higher|lower|greater|less|more|exceed|above|below|larger|smaller)\b',
        r'\b(compare|comparison|differ|difference|ratio)\b',
        r'\b(increase|decrease|decline|rise|drop|fall|grow)\b',
    ]
    if any(re.search(p, q) for p in comparison_patterns):
        caps.append('Quantitative Comparison')

    # Multi-source data fusion
    multi_source_patterns = [
        r'\b(combined|fusion|integrate|multiple|cross-reference)\b',
        r'\b(modis|landsat|sentinel|viirs|aster|srtm|gpm|trmm|era5|chirps)\b.*\b(modis|landsat|sentinel|viirs|aster|srtm|gpm|trmm|era5|chirps)\b',
    ]
    if any(re.search(p, q) for p in multi_source_patterns):
        caps.append('Multi-Source Fusion')

    # Anomaly / event detection
    anomaly_patterns = [
        r'\b(anomal|unusual|extreme|outbreak|event|disaster|flood|fire|drought|storm|eruption|earthquake)\b',
        r'\b(hotspot|burn|wildfire|blaze)\b',
    ]
    if any(re.search(p, q) for p in anomaly_patterns):
        caps.append('Anomaly/Event Detection')

    # Spatial aggregation (reducing over regions)
    aggregation_patterns = [
        r'\b(average|mean|total|sum|median|aggregate|overall)\b',
        r'\b(concentration|density|amount|volume|depth|thickness)\b',
    ]
    if any(re.search(p, q) for p in aggregation_patterns):
        caps.append('Spatial Aggregation')

    # Threshold-based classification
    threshold_patterns = [
        r'\b(threshold|classify|classification|detect|detection|mask|filter)\b',
        r'\b(above|below|exceed|greater than|less than)\s+\d',
    ]
    if any(re.search(p, q) for p in threshold_patterns):
        caps.append('Threshold Classification')

    if not caps:
        caps.append('General EO Query')

    return caps

# Apply to all questions
df_data['capabilities'] = df_data['Question'].apply(classify_capabilities)
df_data['n_capabilities'] = df_data['capabilities'].apply(len)

print('=== Capability Distribution ===')
cap_counter = Counter()
for caps in df_data['capabilities']:
    for c in caps:
        cap_counter[c] += 1

for cap, cnt in cap_counter.most_common():
    print(f'  {cap:35s}: {cnt:4d} ({cnt/len(df_data)*100:.1f}%)')

print(f'\n=== Required Capabilities per Question ===')
print(df_data['n_capabilities'].describe())
print(f'\nQuestions requiring 2+ capabilities: {(df_data["n_capabilities"] >= 2).sum()} ({(df_data["n_capabilities"] >= 2).sum()/len(df_data)*100:.1f}%)')
print(f'Questions requiring 3+ capabilities: {(df_data["n_capabilities"] >= 3).sum()} ({(df_data["n_capabilities"] >= 3).sum()/len(df_data)*100:.1f}%)')

In [ ]:
# Topic × Capability cross-tabulation
print('=== Topic × Capability Matrix ===')
print()

topics = df_data['Tag'].unique()
caps_list = [c for c, _ in cap_counter.most_common()]

# Build cross-tab
cross_tab = {}
for topic in sorted(topics):
    subset = df_data[df_data['Tag'] == topic]
    cross_tab[topic] = {}
    for cap in caps_list:
        count = sum(1 for caps in subset['capabilities'] if cap in caps)
        cross_tab[topic][cap] = count

cross_df = pd.DataFrame(cross_tab).T
print(cross_df.to_string())
print()
print('This shows the two-level taxonomy: Topic (rows) × Capability (columns)')